# 해충 탐지(Pests) YOLO 학습 — Google Colab
클래스: **Aphids · Thrips · Whiteflies** (Roboflow COCO 데이터셋)

실행 전: 메뉴 **런타임 → 런타임 유형 변경 → GPU(T4)** 선택

## 1. 설치 & GPU 확인

In [ ]:
!pip -q install ultralytics
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. 데이터셋 준비
**방법 A** — GitHub Release 에서 다운로드(권장):

In [ ]:
!curl -L -o Pests.v3i.coco.zip https://github.com/givememonkblanc/smartfarm-pest-detection/releases/download/v1.0/Pests.v3i.coco.zip
!mkdir -p Pests && unzip -q Pests.v3i.coco.zip -d Pests && rm Pests.v3i.coco.zip
!ls Pests

**방법 B** — 좌측 파일창에 `Pests.v3i.coco.zip` 을 직접 업로드 후:
```python
!mkdir -p Pests && unzip -q Pests.v3i.coco.zip -d Pests
```

## 3. COCO → YOLO 변환 (data.yaml 생성)

In [ ]:
import json, shutil
from pathlib import Path

ROOT = Path('Pests'); SPLITS = ['train','valid','test']; EXT = {'.jpg','.jpeg','.png'}

# 실제 등장하는 카테고리만 클래스로 채택 (id 오름차순 -> 0..n-1)
used, id2name = set(), {}
for s in SPLITS:
    j = ROOT/s/'_annotations.coco.json'
    if not j.exists(): continue
    d = json.loads(j.read_text())
    for c in d['categories']: id2name[c['id']] = c['name']
    for a in d['annotations']: used.add(a['category_id'])
ids = sorted(used); id2cls = {c:i for i,c in enumerate(ids)}; names = [id2name[c] for c in ids]
print('클래스:', names, '| 매핑:', id2cls)

for s in SPLITS:
    sd = ROOT/s; j = sd/'_annotations.coco.json'
    if not j.exists(): continue
    d = json.loads(j.read_text()); imgs = {im['id']:im for im in d['images']}
    labels = {i:[] for i in imgs}
    for a in d['annotations']:
        cl = id2cls.get(a['category_id'])
        if cl is None: continue
        im = imgs[a['image_id']]; w,h = im['width'],im['height']; x,y,bw,bh = a['bbox']
        cx,cy,nw,nh = (x+bw/2)/w,(y+bh/2)/h,bw/w,bh/h
        labels[a['image_id']].append(f"{cl} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    (sd/'images').mkdir(exist_ok=True); (sd/'labels').mkdir(exist_ok=True)
    for p in list(sd.iterdir()):
        if p.is_file() and p.suffix.lower() in EXT: shutil.move(str(p), sd/'images'/p.name)
    name2id = {Path(im['file_name']).name:i for i,im in imgs.items()}
    for ip in (sd/'images').iterdir():
        i = name2id.get(ip.name)
        (sd/'labels'/f'{ip.stem}.txt').write_text('\n'.join(labels.get(i,[])) if i is not None else '')
    print(s, len(list((sd/'images').iterdir())), 'images')

(ROOT/'data.yaml').write_text(
    f"path: {ROOT.resolve()}\ntrain: train/images\nval: valid/images\ntest: test/images\n\nnc: {len(names)}\nnames: {names}\n")
print(open(ROOT/'data.yaml').read())

### (선택) 클래스 불균형 보정 — train 오버샘플링
Aphids 라벨이 압도적으로 많아, 소수 클래스(Thrips·Whiteflies)가 든 이미지를 복제해 비율을 보정합니다.
**보정이 필요 없으면 이 셀은 건너뛰고 바로 4번(학습)으로 가세요.**

In [ ]:
import math, shutil
from collections import Counter
from pathlib import Path

TAG, MAX_DUP = '.os', 4          # 복제 표식 / 이미지당 최대 복제 배수
tr = Path('Pests/train'); img_d, lbl_d = tr/'images', tr/'labels'

# 재실행 안전: 기존 복제본 정리
for d in (img_d, lbl_d):
    for p in d.glob(f'*{TAG}*'): p.unlink()

def counts():
    c = Counter()
    for t in lbl_d.glob('*.txt'):
        for ln in t.read_text().splitlines():
            if ln.strip(): c[int(ln.split()[0])] += 1
    return c

before = counts(); target = max(before.values())
weight = {k: target/v for k, v in before.items()}
made = 0
for lbl in list(lbl_d.glob('*.txt')):
    if TAG in lbl.name: continue
    cls = {int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip()}
    if not cls: continue
    factor = min(MAX_DUP, math.ceil(max(weight[c] for c in cls)))
    img = next((p for p in img_d.glob(f'{lbl.stem}.*')), None)
    if img is None: continue
    for k in range(1, factor):
        s = f'{lbl.stem}{TAG}{k}'
        shutil.copy(img, img_d/f'{s}{img.suffix}'); shutil.copy(lbl, lbl_d/f'{s}.txt'); made += 1

after = counts()
print(f'복제 {made}장 추가 · train 이미지 {len(list(img_d.glob("*.jpg")))}장')
for c in sorted(after): print(f'  {names[c]:<12} {before[c]:>6} -> {after[c]:>6}')

## 4. 학습

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')   # 더 정확히: yolo11s.pt
model.train(data='Pests/data.yaml', epochs=100, imgsz=640, batch=16, name='pests', patience=20)

## 5. 검증 & 추론

In [ ]:
m = model.val(split='test')
print('mAP50:', round(m.box.map50,4), '| mAP50-95:', round(m.box.map,4))

res = model.predict('Pests/test/images', conf=0.25, save=True)
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('runs/detect/predict*/*.jpg'))[:3]:
    display(Image(f, width=480))

## 6. 가중치 저장 (다운로드)
`best.pt` 를 내려받아 강의 자료의 모델 선택 드롭다운에 연결하세요.

In [ ]:
from google.colab import files
files.download('runs/detect/pests/weights/best.pt')